# Delhi/NCR Rental Market Intelligence - Exploratory Data Analysis
This notebook visualizes and analyzes the cleaned residential rental market dataset to uncover key pricing drivers across Delhi/NCR.
### Analytical Focus Areas:
1. Pricing distributions and descriptive statistics
2. Geospatial trends (rents and rent/sqft across zones and cities)
3. Property characteristic drivers (BHK count, age, size)
4. Connectivity correlations (metro and office hub proximity vs rent)
5. Price premiums for amenities (AC, parking, furnishing)


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
df = pd.read_csv('../data/processed/rental_listings_cleaned.csv')
print(f'Dataset shape: {df.shape}')


Dataset shape: (838, 48)


### 1. Market Overview & Distribution Statistics
Let's look at the average and median rent values and the overall distribution.


In [1]:
print('Overall Rental Statistics:')
stats = df[['monthly_rent', 'area_sqft', 'rent_per_sqft']].agg(['mean', 'median', 'min', 'max', 'std'])
print(stats.round(2))

# Plot rent distribution
plt.figure(figsize=(10, 5))
sns.histplot(df['monthly_rent'], bins=30, kde=True, color='indigo')
plt.title('Distribution of Monthly Rent in Delhi/NCR')
plt.xlabel('Monthly Rent (₹)')
plt.ylabel('Count')
plt.axvline(df['monthly_rent'].median(), color='red', linestyle='--', label=f'Median: ₹{df["monthly_rent"].median():,}')
plt.legend()
plt.show()


### 2. Location-Based Analysis
Analyze rental rates across different cities and localities. We contrast Delhi, Noida, Gurugram, and Ghaziabad.


In [1]:
city_stats = df.groupby('city')[['monthly_rent', 'rent_per_sqft', 'area_sqft']].median().sort_values(by='monthly_rent', ascending=False)
print('Median values by City:')
print(city_stats)

# Median Rent by Locality (Top 10 and Bottom 10)
locality_stats = df.groupby('locality')['monthly_rent'].median().sort_values()
print('\n5 Cheapest Localities (Median Rent):')
print(locality_stats.head(5))
print('\n5 Most Expensive Localities (Median Rent):')
print(locality_stats.tail(5))


Median values by City:
               monthly_rent  rent_per_sqft  area_sqft
city                                                 
Gurugram            31000.0          28.49     1080.0
Delhi               20000.0          22.83      935.0
Noida               17000.0          15.65     1080.0
Ghaziabad           15000.0          14.07     1080.0
Greater Noida       11500.0          10.82     1080.0

5 Cheapest Localities (Median Rent):
locality
Knowledge Park     9500.0
Pari Chowk        11500.0
Shakarpur         11500.0
Laxmi Nagar       12000.0
Indirapuram       15000.0
Name: monthly_rent, dtype: float64

5 Most Expensive Localities (Median Rent):
locality
Civil Lines         45000.0
Greater Kailash     55000.0
Green Park          57500.0
Hauz Khas           58000.0
Golf Course Road    62500.0
Name: monthly_rent, dtype: float64


### 3. Key Price Drivers: Proximity to Metro & Size
We check if metro distance correlates with monthly rent. Note that correlation does not mean causation, but it reveals important spatial sorting.


In [1]:
correlation = df[['monthly_rent', 'area_sqft', 'metro_distance_km', 'office_distance_km', 'property_age']].corr()
print('Correlation Matrix with Rent:')
print(correlation['monthly_rent'].round(3))

# Proximity analysis: Metro distance vs Rent
df['metro_proximity_bucket'] = pd.cut(df['metro_distance_km'], bins=[0, 0.5, 1.0, 2.0, 5.0], labels=['<500m', '500m-1km', '1km-2km', '>2km'])
metro_impact = df.groupby('metro_proximity_bucket')['monthly_rent'].median()
print('\nMedian Rent by Metro Proximity:')
print(metro_impact)


Correlation Matrix with Rent:
monthly_rent         1.000
area_sqft            0.793
metro_distance_km   -0.084
office_distance_km   -0.015
property_age        -0.076
Name: monthly_rent, dtype: float64

Median Rent by Metro Proximity:
metro_proximity_bucket
<500m       25000.0
500m-1km    20000.0
1km-2km     18500.0
>2km        17500.0
Name: monthly_rent, dtype: float64


### 4. Amenity Rental Premiums
Calculate how much extra rent properties with parking, AC, and high-speed WiFi command compared to properties without.


In [1]:
def calculate_premium(col_name):
    group = df.groupby(col_name)['monthly_rent'].median()
    yes_val = group.get('Yes', 0)
    no_val = group.get('No', 0)
    diff = yes_val - no_val
    pct = (diff / no_val * 100) if no_val > 0 else 0
    return yes_val, no_val, diff, pct

for amenity in ['ac', 'parking', 'wifi', 'power_backup']:
    yes, no, diff, pct = calculate_premium(amenity)
    print(f'{amenity.upper()} Premium: Yes=₹{yes:,}, No=₹{no:,} | Premium=₹{diff:,} ({pct:.1f}%)')


AC Premium: Yes=₹25,000, No=₹18,000 | Premium=₹7,000 (38.9%)
PARKING Premium: Yes=₹28,500, No=₹15,000 | Premium=₹13,500 (90.0%)
WIFI Premium: Yes=₹20,000, No=₹20,000 | Premium=₹0 (0.0%)
POWER_BACKUP Premium: Yes=₹22,000, No=₹18,000 | Premium=₹4,000 (22.2%)
